<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A4_Programming_Cross_Entropy_Loss_on_GPU_(COS484_S2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Programming in Problem 4
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

## Cross Entropy Loss on GPU
In this problem you will benchmark the cross-entropy loss in PyTorch and measure its efficiency relative to the theoretical memory bandwidth of the GPU. You will predict performance before measuring, then explain any discrepancies — this is the core skill of systems performance analysis.
Setup: Use the following parameters throughout:

-  Batch size: B = 4,096
-  Vocabulary size: V = 128,256
-  Logits dtype: torch.bfloat16
-  Reduction: ’none’
- Device: GPU (use a Colab GPU runtime, e.g., T4 or A100)
-  Report your PyTorch version (e.g., torch. version ) and GPU model in your notebook. Results can vary significantly across versions.

## Generate random input logits and targets

In [ ]:
# Generate random input logits and targets:
import torch
B, V = 4096, 128256
logits = torch.randn(B, V, dtype=torch.bfloat16 , device=’cuda’,
requires_grad=True)
targets = torch.randint(0, V, (B,), device=’cuda’)

# Questions

**(a) Predict before you measure. (8 points)**

Before writing any benchmarking code, use your answers from Theory Problem 1 to predict the performance of an ideal (perfectly memory-bound) cross-entropy kernel on your GPU.

(i) Look up the peak memory bandwidth of your Colab GPU. For this assignment, if you are using an A100, assume the A100 SXM bandwidth of ∼2039 GB/s; if you are using a T4, use ∼300 GB/s. Using the total bytes from Theory Problem 1(c), compute the minimum possible time (in ms) for the forward pass and backward pass, assuming the kernel runs at 100% of peak memory bandwidth. (4 points)

(ii) Based on your arithmetic intensity analysis from Theory Problem 1(c), explain why memory bandwidth (not TFLOPS) is the right performance limiter and the right metric to evaluate the efficiency of cross- entropy. (4 points)



TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)


**(b) Benchmark PyTorch eager mode. (12 points)**


(i) Write a benchmarking function that measures the wall-clock time (in milliseconds) for the cross-entropy forward pass and backward pass separately. Use torch.cuda.Event with enable timing=True for accurate GPU timing. After recording the end event, make sure to synchronize the GPU (e.g., with torch.cuda.synchronize() or end event.synchronize()) before reading the elapsed time. Run each measurement multiple times (e.g., 100 iterations) and report the median time. Make sure to do a few warmup iterations before timing. (5 points)


(ii) Compute the achieved memory bandwidth (in GB/s) for the forward and backward passes: Achieved BW (GB/s) = Total bytes (GB) / Time (s)
    
How does the measured time compare to your prediction from part (a)? If there is a significant gap, explain why PyTorch eager mode is slower than the ideal. What extra memory traffic or overhead might the eager implementation incur? (7 points)

Hint: Think about whether the eager implementation fuses all the operations into a single kernel, or whether it materializes intermediate tensors (e.g., the full softmax output) to HBM.

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)




Code for part (b):

**(c) Benchmark torch.compile mode. (10 points)**

(i) Wrap the cross-entropy computation with torch.compile and repeat the benchmarking from part (b). Note that the first call triggers compilation — make sure your warmup accounts for this. Report the achieved memory bandwidth. (4 points)


(ii) How does torch.compile compare to eager mode and to your ideal prediction? Explain what torch.compile is likely doing differently (e.g., kernel fusion) that accounts for the performance difference. (6 points)

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)



Code for part (c):


**(d) Inspecting the compiled kernel. (10 points)**

When torch.compile compiles your function, it generates Triton kernel code under the hood.

(i) Use the environment variable TORCH LOGS=output code or torch. dynamo.config.log level to dump the generated Triton kernel code for the cross-entropy forward pass. Include the generated code (or relevant excerpts) in your notebook. (3 points)


(ii) Annotate the generated kernel: identify which lines correspond to (1) computing the max for numer- ical stability, (2) the exp and sum (softmax denominator), (3) the log-sum-exp, and (4) the final loss computation. Relate these back to the formulas you derived in Theory Problem 1(a). (7 points)

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)



Code for part (d):

# Problem 2: Bonus — Cross-Entropy Speed Competition

See task specification and submission instruction in the problem set PDF. Submission window is open from April 7 to April 16. If you're submitting for bonus credits, please include your optimization journal here documenting your optimization process:


- What approaches did you try? (e.g., torch.compile tricks, custom Triton kernels, fused forward+backward,
memory layout changes, block size tuning, etc.)
- For each approach, report the timing and explain why it was faster or slower than your previous best.
- What is the achieved memory bandwidth of your final submission, and what fraction of peak A100 bandwidth (2039 GB/s) does it reach? What do you think is preventing you from reaching 100%?


If you prefer to document your optimization in a separate PDF, please also make a note here.


# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   
*   



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)